{ “cells”: \[ { “cell_type”: “markdown”, “metadata”: {}, “source”: \[
“\# Gen Z Consumer Insights - Exploratory Data Analysis”, “”, “Hi! This
is my EDA notebook for a synthetic 5,000-respondent Gen Z consumer
survey (`data/survey_responses.csv`). The main goal here is to find real
patterns in the survey data and then sanity-check them against \~50
actual cited industry stats (`data/industry_benchmarks.csv`), so I’m not
just making claims that only exist inside a made-up dataset. I’m also
trying to be pretty upfront about where the data is *not* saying much of
anything, because I think that’s just as important to report as the
interesting stuff.”, “”, “This started as a class project but I’ve
cleaned it up enough that I want to put it on GitHub as a portfolio
piece, so I tried to actually document my reasoning instead of just
dumping charts.”, “”, “**Other files in this repo, in case you’re
browsing around:**”, “- `app.py` - the same analysis, but as an
interactive Streamlit dashboard you can filter live (no code needed)”,
“- `src/data_utils.py` - the loading/cleaning helper functions shared
between this notebook and the app”, “”, “**What’s in this notebook**”,
“1. [Setup & data overview](#1)”, “2. [Demographics](#2)”, “3. [Money &
spending](#3)”, “4. [Media, screen time & platforms](#4)”, “5. [Values &
trust](#5)”, “6. [Correlation check (and why it matters here)](#6)”, “7.
[Key findings](#7)”, “8. [Anomalies & data quality notes](#8)”, “9.
[Broader context - outside cited research](#9)” \] }, { “cell_type”:
“markdown”, “metadata”: {}, “source”: \[ “<a id='1'></a>”, “\## 1. Setup
& data overview” \] }, { “cell_type”: “code”, “execution_count”: null,
“metadata”: {}, “outputs”: \[\], “source”: \[ “import sys”,
“sys.path.append(‘..’) \# so I can import from src/ while running from
notebooks/”, “”, “import pandas as pd”, “import numpy as np”, “import
matplotlib.pyplot as plt”, “import seaborn as sns”, “”, “from
src.data_utils import load_survey, load_benchmarks, trust_gap_summary”,
“”, “sns.set_theme(style="whitegrid", palette="deep")”,
“plt.rcParams\["figure.figsize"\] = (9, 5)”,
“pd.set_option("display.max_columns", 30)”, “”, “survey =
load_survey()”, “benchmarks = load_benchmarks()”, “”, “print(f"Survey:
{survey.shape\[0\]:,} respondents x {survey.shape\[1\]} columns")”,
“print(f"Benchmarks: {benchmarks.shape\[0\]} cited external stats across
{benchmarks.category.nunique()} categories")”, “survey.head()” \] }, {
“cell_type”: “code”, “execution_count”: null, “metadata”: {}, “outputs”:
\[\], “source”: \[ “survey.info()” \] }, { “cell_type”: “code”,
“execution_count”: null, “metadata”: {}, “outputs”: \[\], “source”: \[
“\# checking nulls once up front so I don’t have to keep second-guessing
myself later”, “survey.isnull().sum().to_frame("missing_values").T” \]
}, { “cell_type”: “markdown”, “metadata”: {}, “source”: \[ “Everything’s
fully populated except `discretionary_pct_of_income`, which is missing
for 63 rows out of 5,000 (about 1.3%). I looked into it a little and it
lines up with respondents who reported
$0 monthly income, so the percentage calc is undefined for them (divide by zero situation), not a data entry mistake. I'm leaving those rows as NaN rather than imputing something fake, and just excluding them from any mean/median calc on that column, which pandas does automatically anyway."  ]  },  {  "cell_type": "code",  "execution_count": null,  "metadata": {},  "outputs": [],  "source": [  "survey.describe().T"  ]  },  {  "cell_type": "markdown",  "metadata": {},  "source": [  "<a id='2'></a>\n",  "## 2. Demographics"  ]  },  {  "cell_type": "code",  "execution_count": null,  "metadata": {},  "outputs": [],  "source": [  "fig, axes = plt.subplots(2, 2, figsize=(13, 9))\n",  "\n",  "survey.region.value_counts().plot(kind=\"bar\", ax=axes[0,0], color=\"#7C3AED\")\n",  "axes[0,0].set_title(\"Region\"); axes[0,0].tick_params(axis='x', rotation=0)\n",  "\n",  "survey.gender.value_counts().plot(kind=\"bar\", ax=axes[0,1], color=\"#7C3AED\")\n",  "axes[0,1].set_title(\"Gender\"); axes[0,1].tick_params(axis='x', rotation=30)\n",  "\n",  "survey.education.value_counts().plot(kind=\"bar\", ax=axes[1,0], color=\"#7C3AED\")\n",  "axes[1,0].set_title(\"Education\"); axes[1,0].tick_params(axis='x', rotation=20)\n",  "\n",  "survey.employment.value_counts().plot(kind=\"bar\", ax=axes[1,1], color=\"#7C3AED\")\n",  "axes[1,1].set_title(\"Employment\"); axes[1,1].tick_params(axis='x', rotation=20)\n",  "\n",  "plt.tight_layout()\n",  "plt.savefig(\"../assets/demographics_overview.png\", dpi=130, bbox_inches=\"tight\")\n",  "plt.show()"  ]  },  {  "cell_type": "markdown",  "metadata": {},  "source": [  "Quick notes on what I'm looking at here:\n",  "\n",  "- Ages run 18-28, which is basically the full \"adult Gen Z\" range (born roughly 1997-2007), not just teenagers, so keep that in mind for anything that sounds like it should apply to high schoolers specifically - it doesn't necessarily.\n",  "- Region and gender both look like a pretty reasonable spread, nothing that jumps out as obviously skewed or synthetic-looking on its own.\n",  "- I keep coming back to employment status as the main lens through the rest of this notebook, mostly because it turned out to be the single strongest real predictor of income in the data (see section 3), way more than education or region."  ]  },  {  "cell_type": "markdown",  "metadata": {},  "source": [  "<a id='3'></a>\n",  "## 3. Money & spending"  ]  },  {  "cell_type": "code",  "execution_count": null,  "metadata": {},  "outputs": [],  "source": [  "income_by_employment = survey.groupby(\"employment\")[\"annual_income_usd\"].agg([\"mean\",\"median\",\"count\"]).sort_values(\"mean\")\n",  "income_by_employment.style.format({\"mean\": \"${:,.0f}",
"median":
"${:,.0f}\"})"  ]  },  {  "cell_type": "markdown",  "metadata": {},  "source": [  "This is basically the expected ordering - unemployed lowest, full-time highest, student and gig/freelance sitting in between - so nothing surprising here, but it's a good sanity check that the synthetic data is internally consistent before I go trusting the more interesting stuff downstream."  ]  },  {  "cell_type": "code",  "execution_count": null,  "metadata": {},  "outputs": [],  "source": [  "fig, ax = plt.subplots()\n",  "sample = survey.sample(1500, random_state=42)\n",  "sns.scatterplot(data=sample, x=\"annual_income_usd\", y=\"monthly_discretionary_usd\",\n",  " hue=\"employment\", alpha=0.5, ax=ax, palette=\"Set2\")\n",  "ax.set_title(\"Income vs. monthly discretionary spend\")\n",  "ax.set_xlabel(\"Annual income ($)");
ax.set_ylabel("Monthly discretionary spend (\$)")”,
“plt.tight_layout()”,
“plt.savefig("../assets/income_vs_discretionary.png", dpi=130,
bbox_inches="tight")”, “plt.show()”, “”, “print(f"Correlation (income vs
discretionary spend): r =
{survey.annual_income_usd.corr(survey.monthly_discretionary_usd):.3f}")”
\] }, { “cell_type”: “code”, “execution_count”: null, “metadata”: {},
“outputs”: \[\], “source”: \[ “discretionary_pct_by_employment =
survey.groupby("employment")\["discretionary_pct_of_income"\].mean().sort_values(ascending=False)”,
“print("Discretionary spend as % of income, by employment status:")”,
“discretionary_pct_by_employment.round(1)” \] }, { “cell_type”:
“markdown”, “metadata”: {}, “source”: \[ “Okay this is the first finding
I’d actually call interesting. Discretionary spend scales almost
perfectly with income - r is like 0.88, so not shocking on its own - but
as a *share* of income it’s basically flat, sitting around 22% no matter
whether someone is unemployed, a student, part-time, gig/freelance, or
full-time. Full range is only 22.2% to 22.5% across all five employment
categories, which is a tighter spread than I expected going in.”, “”,
“So my read on this: employment status changes *how much* people earn a
lot, but in this dataset it does not really change the *proportion* of
income someone treats as "fun money." That’s kind of a counterintuitive
result if you assume people with less stable income (gig workers,
students) would be more cautious with discretionary spend as a
percentage, and this data doesn’t support that assumption at all.” \] },
{ “cell_type”: “code”, “execution_count”: null, “metadata”: {},
“outputs”: \[\], “source”: \[ “bnpl_by_income_q =
pd.crosstab(survey.income_quartile, survey.uses_buy_now_pay_later,
normalize="index") \* 100”, “bnpl_by_income_q.round(1)” \] }, {
“cell_type”: “code”, “execution_count”: null, “metadata”: {}, “outputs”:
\[\], “source”: \[ “fig, ax = plt.subplots()”,
“bnpl_by_income_q\["Yes"\].plot(kind="bar", ax=ax, color="#F97316")”,
“ax.set_title("BNPL adoption (%) by income quartile")”,
“ax.set_ylabel("% using Buy Now, Pay Later")”, “ax.set_xlabel("Income
quartile")”, “plt.xticks(rotation=0)”, “plt.tight_layout()”,
“plt.savefig("../assets/bnpl_by_income.png", dpi=130,
bbox_inches="tight")”, “plt.show()” \] }, { “cell_type”: “markdown”,
“metadata”: {}, “source”: \[ “BNPL (Buy Now Pay Later) usage lands in a
tight 41-43% band across every single income quartile - lowest earners
aren’t meaningfully more reliant on it than the highest earners in this
survey. That’s worth flagging because I think the default assumption
most people have (myself included, before I ran this) is that BNPL is
primarily a tool for people who can’t otherwise access credit. Here it
reads a lot more like a general payment-method preference that Gen Z as
a whole leans into, rather than something concentrated in the lower
income group specifically. I’d want a bigger real-world dataset before
I’d fully trust this, but it’s consistent enough within this survey that
I don’t think it’s random noise.” \] }, { “cell_type”: “markdown”,
“metadata”: {}, “source”: \[ “<a id='4'></a>”, “\## 4. Media, screen
time & platforms” \] }, { “cell_type”: “code”, “execution_count”: null,
“metadata”: {}, “outputs”: \[\], “source”: \[ “fig, ax =
plt.subplots()”, “sns.histplot(survey.daily_screen_hours, bins=40,
color="#7C3AED", ax=ax)”, “ax.axvline(survey.daily_screen_hours.mean(),
color="#F97316", linestyle="–",”, ” label=f"mean =
{survey.daily_screen_hours.mean():.1f}h")“,”ax.set_title("Daily screen
time distribution")“,”ax.set_xlabel("Hours /
day")“,”ax.legend()“,”plt.tight_layout()“,”plt.savefig("../assets/screen_time_distribution.png",
dpi=130,
bbox_inches="tight")“,”plt.show()“,”“,”print(survey.daily_screen_hours.describe().round(2))”
\] }, { “cell_type”: “markdown”, “metadata”: {}, “source”: \[ “Roughly
bell-shaped, centered around 6 hours a day, with a pretty long right
tail out to 16 hours for a handful of people. I dig into those outliers
more in section 8 because I don’t fully trust them without more info.”
\] }, { “cell_type”: “code”, “execution_count”: null, “metadata”: {},
“outputs”: \[\], “source”: \[ “screen_by_platform =
survey.groupby("primary_platform")\["daily_screen_hours"\].agg(\["mean","count"\]).sort_values("mean",
ascending=False)”, “screen_by_platform.round(2)” \] }, { “cell_type”:
“markdown”, “metadata”: {}, “source”: \[ “Average daily screen time sits
in a really narrow 5.9-6.2 hour band no matter which platform someone
names as their primary one. Reddit and Snapchat users actually
self-report marginally *more* time than TikTok users, which honestly
goes against the pretty common assumption that TikTok specifically is
the biggest time-sink. The spread here is small enough (only about 0.3
hours between the highest and lowest platform) that I’d call this
basically "flat across platforms" rather than a real difference by
platform.”, “”, “One more thing worth calling out: this survey’s
`daily_screen_hours` (about 6.0h average across the whole panel) runs
noticeably higher than the independently-reported industry figure for
social-media-specific time, which is closer to 4.5h/day per
Cropink/Attest 2025 (see `data/industry_benchmarks.csv`). That actually
makes sense once you think about it, because this survey column is
measuring *total* screen time, not social-media-only time, so the two
numbers aren’t really measuring the same thing and shouldn’t be directly
compared.” \] }, { “cell_type”: “code”, “execution_count”: null,
“metadata”: {}, “outputs”: \[\], “source”: \[ “discovery =
survey.brand_discovery_channel.value_counts(normalize=True).mul(100).round(1)”,
“channel =
survey.preferred_shopping_channel.value_counts(normalize=True).mul(100).round(1)”,
“”, “fig, axes = plt.subplots(1, 2, figsize=(13,5))”,
“discovery.sort_values().plot(kind="barh", ax=axes\[0\],
color="#7C3AED")”, “axes\[0\].set_title("How brands get discovered");
axes\[0\].set_xlabel("% of respondents")”, “”,
“channel.sort_values().plot(kind="barh", ax=axes\[1\],
color="#F97316")”, “axes\[1\].set_title("Preferred shopping channel");
axes\[1\].set_xlabel("% of respondents")”, “”, “plt.tight_layout()”,
“plt.savefig("../assets/discovery_and_channel.png", dpi=130,
bbox_inches="tight")”, “plt.show()” \] }, { “cell_type”: “markdown”,
“metadata”: {}, “source”: \[ “Social media is the single largest
brand-discovery channel at 40.8%, way ahead of search (24.1%),
word-of-mouth (17.8%), paid influencer content (11.0%), and traditional
ads dead last at 6.3%. Mobile apps (39.3%) are also the top preferred
shopping channel, ahead of websites, in-store, and social commerce
specifically. Both of these line up directionally with the outside
benchmark figures I pull in section 9, which makes me a bit more
confident these two patterns are actually reflecting something real
rather than being an artifact of how the synthetic data was generated.”
\] }, { “cell_type”: “markdown”, “metadata”: {}, “source”: \[
“<a id='5'></a>”, “\## 5. Values & trust” \] }, { “cell_type”: “code”,
“execution_count”: null, “metadata”: {}, “outputs”: \[\], “source”: \[
“tg = trust_gap_summary(survey)”, “fig, ax =
plt.subplots(figsize=(7,3.5))”, “ax.barh(tg.channel, tg.avg_trust_1to5,
color=\["#94A3B8", "#7C3AED"\])”, “for i, v in
enumerate(tg.avg_trust_1to5):”, ” ax.text(v + 0.05, i, f"{v:.2f}",
va="center")“,”ax.set_xlim(0, 5)“,”ax.set_title("Avg. trust: traditional
ads vs. influencers (1-5
scale)")“,”plt.tight_layout()“,”plt.savefig("../assets/trust_gap.png",
dpi=130, bbox_inches="tight")“,”plt.show()” \] }, { “cell_type”: “code”,
“execution_count”: null, “metadata”: {}, “outputs”: \[\], “source”: \[
“for group_col in \["region", "gender", "employment",
"income_quartile"\]:”, ” print(f"\n— trust gap by {group_col} —")“,” g =
survey.groupby(group_col)\[\["trust_traditional_ads_1to5","trust_influencers_1to5"\]\].mean()“,”
g\["gap"\] = g\["trust_influencers_1to5"\] -
g\["trust_traditional_ads_1to5"\]“,” print(g.round(2))” \] }, {
“cell_type”: “markdown”, “metadata”: {}, “source”: \[ “This is, in my
opinion, the sharpest and most consistent signal in the entire survey.
Trust in traditional advertising averages 2.33 out of 5, trust in
influencers averages 3.39 out of 5 - a gap of roughly one full point on
a 5-point scale - and here’s the part that actually convinced me it’s
meaningful: that gap is nearly identical across every region, gender,
employment status, and income quartile I sliced it by. I went into this
expecting the gap to be bigger for, say, younger or lower-income
respondents, and it just wasn’t - the gap sits between about 1.00 and
1.14 across every single subgroup I checked, with the sole exception of
the small "prefer not to say" gender group (n=45), where the gap widens
to 1.49, though I wouldn’t put much weight on that given how small that
group is.”, “”, “When a pattern shows up this consistently across
basically every subgroup instead of being concentrated in one slice,
that’s usually a decent sign it reflects something like a genuine
generational baseline rather than a quirk tied to one particular segment
of respondents.” \] }, { “cell_type”: “markdown”, “metadata”: {},
“source”: \[ “<a id='6'></a>”, “\## 6. Correlation check (and why it
matters here)” \] }, { “cell_type”: “code”, “execution_count”: null,
“metadata”: {}, “outputs”: \[\], “source”: \[ “numeric_cols =
\["annual_income_usd","monthly_discretionary_usd","daily_screen_hours",”,
” "val_sustainability_1to5","val_brand_authenticity_1to5",”, ”
"trust_traditional_ads_1to5","trust_influencers_1to5"\]”, “”, “corr =
survey\[numeric_cols\].corr()”, “fig, ax = plt.subplots(figsize=(8,6))”,
“sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdPu", center=0, ax=ax,
square=True)”, “ax.set_title("Correlation matrix - numeric survey
fields")”, “plt.tight_layout()”,
“plt.savefig("../assets/correlation_heatmap.png", dpi=130,
bbox_inches="tight")”, “plt.show()” \] }, { “cell_type”: “markdown”,
“metadata”: {}, “source”: \[ “I want to read this chart carefully,
because I think the mostly-empty heatmap is itself a finding and not
just a failed analysis. Outside of the expected income vs
discretionary-spend relationship (r is about 0.88), literally every
other pairwise correlation among these numeric fields is under +-0.05.
Screen time doesn’t predict trust in influencers. Valuing sustainability
doesn’t predict valuing brand authenticity more or less. None of the
attitude/values fields move together with each other at all.”, “”, “For
an honest writeup, I think that’s worth stating plainly rather than
glossing over: the Likert-scale attitude fields in this survey seem to
behave like independently-drawn distributions layered on top of the
demographics, which is a pretty common thing to see in synthetic or
simulated survey data specifically. That doesn’t make this data
useless - it’s genuinely good for practicing EDA, filtering, and
plotting - but it does mean I should describe the *distributions* I’m
seeing (like the ad-vs-influencer trust gap above, which holds up
because it’s a difference in central tendency between two groups, not a
claimed correlation between two variables) instead of trying to build
"Gen Z segment X thinks Y because of Z" narratives that the correlation
structure here just doesn’t support. I’d rather under-claim than
over-claim on this one.” \] }, { “cell_type”: “markdown”, “metadata”:
{}, “source”: \[ “<a id='7'></a>”, “\## 7. Key findings”, “”, “Pulling
together everything above, here’s what I’d actually put weight behind:”,
“”, “1. **The ad-trust / influencer-trust gap (2.33 vs. 3.39 out of 5)
is the sharpest, most consistent signal in the whole dataset**, holding
steady across region, gender, income, and employment status.”, “2.
**Discretionary spending is a near-fixed \~22% share of income**,
regardless of employment type - income level explains spend almost
perfectly on its own, while employment *type* barely adds anything once
income is already accounted for.”, “3. **BNPL adoption (\~42%) is flat
across income quartiles** - it looks like a broad payment preference in
this data, not a narrowly low-income-specific coping mechanism.”, “4.
**Social media is the dominant brand-discovery channel (40.8%)**,
beating search, word-of-mouth, and paid influencer content combined with
traditional ads.”, “5. **Screen time barely varies by primary platform**
(5.9-6.2h range) - which platform someone prefers doesn’t meaningfully
predict how much time overall they spend online.”, “”, “The common
thread across 1, 3, 4, and 5 is basically: individual demographic slices
matter way less than I expected going in, and a handful of
behaviors/attitudes look like they’re shared broadly across the whole
Gen Z panel rather than concentrated in specific subgroups.” \] }, {
“cell_type”: “markdown”, “metadata”: {}, “source”: \[ “<a id='8'></a>”,
“\## 8. Anomalies & data quality notes”, “”, “Trying to be transparent
here about what I don’t fully trust, instead of just presenting all the
charts above as if they’re bulletproof:”, “”, “- **Near-zero correlation
among the attitudinal variables** (section 6) - flagging this again
explicitly so nobody, including future-me revisiting this later,
over-interprets the findings above as causal relationships that this
data can actually support.”, “- **Screen-time outliers:** 9 respondents
(0.18% of the panel) report screen time 3+ standard deviations above the
mean, up to 16 hours a day. That’s plausible for genuinely extreme
users, but it’s also the kind of thing that could be caused by device
double-counting or always-on background usage getting counted as "screen
time." I’d want to sanity-check this against a different data source
before using this column for anything higher-stakes than exploratory
analysis.”, “- **Small subgroups:** the "Non-binary" (n=92) and "Prefer
not to say" (n=45) gender categories are small enough that any
differences involving them - like the wider trust gap I flagged in
section 5 - should be read as directional at best, not statistically
solid, without a bigger sample size or actual confidence intervals
attached.”, “- **Total screen time vs. social-only time:** the survey’s
`daily_screen_hours` (about 6.0h average) is noticeably higher than the
external, social-media-specific benchmark of \~4.5h/day. Good reminder
to me (and to anyone reading this) to check that two "screen time"
numbers from different sources are actually measuring the same
underlying thing before comparing them directly.” \] }, { “cell_type”:
“code”, “execution_count”: null, “metadata”: {}, “outputs”: \[\],
“source”: \[ “benchmarks.head(10)” \] }, { “cell_type”: “markdown”,
“metadata”: {}, “source”: \[ “<a id='9'></a>”, “\## 9. Broader context -
cited external research”, “”, “The two clearest signals in this survey -
low trust in traditional advertising, and spending that tracks income
tightly rather than identity or vibes - line up with what independent,
real-world research is currently finding about Gen Z as a generation.
All figures below are quoted from and linked to their original reporting
(also logged with sources and years in
`data/industry_benchmarks.csv`):”, “”, “- **Gen Z’s global spending
power is projected to roughly quadruple, from an estimated \$2.7
trillion in 2024 to \$12.6 trillion by 2030**, per the Bank of America
Institute’s *"Gen Z: A new economic force"* report - even as a lot of
individuals within the cohort report real budget pressure right now.
[Bank of America Institute,
2025](https://institute.bankofamerica.com/economic-insights/genz-new-economic-force.html)”,
“”, “- **U.S. Gen Z spending actually fell about 13% between January and
April 2025**, concentrated in apparel, accessories, and electronics,
according to PwC’s analysis of roughly a million consumer transactions -
PwC reads this as a shift toward value-consciousness rather than simple
frugality. [PwC, "Gen Z Consumer Trends,"
2025](https://www.pwc.com/us/en/industries/consumer-markets/library/gen-z-consumer-trends.html)”,
“”, “- **Weekly video-game spending among 18-24 year olds fell roughly
25% year-over-year**, per Circana data reported by the Wall Street
Journal - a much steeper drop than the under-5% decline seen among older
generations over the same period. [via PC Gamer,
2025](https://www.pcgamer.com/gaming-industry/new-study-shows-that-gen-z-is-spending-way-less-money-on-videogames-than-older-gamers/)”,
“”, “- **Gen Z’s average credit card balance was about \$3,493 as of
mid-2025** - the lowest of any adult generation, per Experian, though
it’s growing faster year-over-year than older cohorts’ balances are.
[Experian, "Average Credit Card Debt by Age,"
2025](https://www.experian.com/blogs/ask-experian/research/credit-card-debt-by-age/)”,
“”, “- **85% of Gen Z say social media influences their purchasing
decisions**, per a 2025 ICSC report covered by Retail Dive - well ahead
of traditional advertising, and directly consistent with this survey’s
ad-trust/influencer-trust gap and its social-media-led brand discovery
finding above. [Retail Dive,
2025](https://www.retaildive.com/news/generation-z-social-media-influence-shopping-behavior-purchases-tiktok-instagram/652576/)”,
“”, “**Putting it all together:** both this survey and the outside
research I checked against it point to a generation with genuinely large
*aggregate* spending power and a structurally low baseline of trust in
traditional marketing - but one that is, in practice, currently spending
more cautiously than its long-run purchasing power alone would suggest.
For anyone thinking about how to reach this group, that combination
seems to argue for earning attention through credible peer- and
creator-level channels rather than traditional advertising, while also
not assuming today’s caution is some kind of permanent ceiling on Gen Z
spend going forward.”, “”, “*One more note on the benchmark dataset
itself:* a handful of rows in `data/industry_benchmarks.csv` include a
`notes` field flagging places where different sources actually disagree
with each other (QSR visit trend figures especially have directly
conflicting numbers depending on which 2025 report you’re looking at). I
left that disagreement in on purpose rather than picking a winner, since
papering over conflicting real-world estimates felt less honest than
just showing the range and letting the reader decide.” \] } \],
“metadata”: { “kernelspec”: { “display_name”: “Python 3”, “language”:
“python”, “name”: “python3” }, “language_info”: { “codemirror_mode”: {
“name”: “ipython”, “version”: 3 }, “file_extension”: “.py”, “mimetype”:
“text/x-python”, “name”: “python”, “nbconvert_exporter”: “python”,
“pygments_lexer”: “ipython3”, “version”: “3.12.3” } }, “nbformat”: 4,
“nbformat_minor”: 5 }